# Milestone 2 Pushshift Reddit DATA ONLY

This notebook performs exploratory data analysis on a Pushshift Reddit dataset stored as
Parquet files on Expanse. The dataset covers Reddit submissions from 2012–2018 across
all subreddits. We examine schema, distributions, data quality, and temporal trends.

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from datetime import datetime

# 16 CPUS / 128 GB Memory
spark = SparkSession.builder \
    .appName("PushshiftRedditEDA") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.executor.memory", "16g") \
    .config("spark.executor.instances", 6) \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Spark version: 3.5.0
Spark UI: http://exp-1-09.expanse.sdsc.edu:4041


## 1. Data Loading

We read all Parquet files from `DATA_DIR` and union them into a single Spark DataFrame.

In [6]:
import os
import glob
from pyspark.sql import functions as F

DATA_DIR = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/"
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.parquet")))

COLS = ["author", "created_utc", "id", "num_comments", "score", 
        "selftext", "subreddit", "subreddit_id", "title"]


In [7]:
pre_cutoff = [f for f in files if os.path.basename(f) >= "RS_2015-01"]
post_cutoff = [f for f in files if os.path.basename(f) < "RS_2015-01"]

# Files where created_utc is STRING - need to cast
df_pre = spark.read.parquet(*pre_cutoff) \
    .select(*[F.col(c) for c in COLS if c != 'created_utc'],
            F.col('created_utc').cast('long').alias('created_utc'))

df_post = spark.read.parquet(*post_cutoff) \
    .select(*[F.col(c) for c in COLS if c != 'created_utc'],
            F.col('created_utc').cast('long').alias('created_utc'))

MIN_TS = 1119398400  # June 2005
MAX_TS = 1700000000  # Nov 2023

df = df_pre.union(df_post) \
    .filter(F.col('created_utc').between(MIN_TS, MAX_TS))

print(f"Files loaded: {len(files)}")
print(f"Partitions: {df.rdd.getNumPartitions()}")


Files loaded: 218
Partitions: 683


In [8]:
df.select(
    F.count('created_utc').alias('non_null'),
    F.from_unixtime(F.min('created_utc')).alias('min_utc'),
    F.from_unixtime(F.max('created_utc')).alias('max_utc')
).show()

+---------+-------------------+-------------------+
| non_null|            min_utc|            max_utc|
+---------+-------------------+-------------------+
|549662955|2012-01-01 00:00:01|2018-12-31 23:59:59|
+---------+-------------------+-------------------+

